In [1]:
import pandas as pd

df = pd.read_csv("google_review_ratings.csv")

# Drop non-feature columns
df = df.drop(columns=["User", "Unnamed: 25"], errors="ignore")

# Convert to numeric
df = df.apply(pd.to_numeric, errors="coerce")

# Fill missing values
df = df.fillna(df.median())

# Create average rating per user
df["avg_rating"] = df.mean(axis=1)

# Create target variable using median threshold
threshold = df["avg_rating"].median()
df["target"] = (df["avg_rating"] >= threshold).astype(int)

# Separate features and target
X = df.drop(columns=["avg_rating", "target"])
y = df["target"]

print("Threshold used:", threshold)
print(y.value_counts())

Threshold used: 2.0156250000000004
target
0    2728
1    2728
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(4092, 24) (1364, 24)


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled)
print(X_test_scaled)

[[-0.04373115 -0.60561056 -0.82398204 ... -0.22305004 -0.11192165
  -0.13456332]
 [ 0.70890355 -0.19324115 -0.35682707 ...  2.01716785  2.62212728
   0.06441796]
 [-0.79636584 -1.04593722 -0.76855687 ... -0.61508817 -0.58018418
  -0.67094765]
 ...
 [ 0.72085013 -0.18625184 -0.38058072 ...  0.20632505  0.43186708
   0.51428869]
 [-0.55743419  1.8755952  -0.36474495 ... -0.57152838 -0.45178961
  -0.48061773]
 [ 0.99562152 -0.02549767 -0.15888006 ...  0.38056422  0.64334048
   2.98857768]]
[[-1.74014585 -1.05991584 -0.79231052 ... -1.09424589 -1.1541834
  -1.33710238]
 [-0.10346406 -0.6265785  -0.84773568 ... -0.24171852 -0.13457952
  -0.1605174 ]
 [ 0.44607873 -0.17926253 -0.33307343 ...  2.01716785  2.62212728
   0.08172068]
 ...
 [-0.13930381 -0.67550368 -0.75272111 ... -0.78310451 -0.3762634
  -0.1605174 ]
 [-0.31850254  1.8755952  -0.04011184 ... -0.5528599  -0.42157913
  -0.44601229]
 [ 1.41375191  0.23310688 -0.03219396 ...  1.17086331  1.64783912
   0.95550805]]


In [17]:
import numpy as np

def knn(X, y):
    return X, y

def prediction(X_test, X_train, y_train, k=3):
    final_preds = []
    
    for x_test in X_test:
        # distances variable holds the Euclidean distance calculation
        euclidiean_dist = np.sqrt(np.sum((X_train - x_test) ** 2, axis=1))
        
        #Sorting the array from smallest to largest and then taking the first k_index
        k_index = np.argsort(euclidiean_dist)[:k]
        k_labels = y_train[k_index]
      
        unique, counts = np.unique(k_labels, return_counts=True) # count how many unique value appears
        pred_value = unique[np.argmax(counts)] # index of max count
        final_preds.append(pred_value) # saving this prediction value
    
    return np.array(final_preds)

In [18]:
# Train model (just stores data)
X_train_stored, y_train_stored = knn(X_train_scaled, y_train.values)

# Predict
y_pred = predict(X_test_scaled, X_train_stored, y_train_stored, k=3)

# Accuracy
accuracy = np.mean(y_pred == y_test.values)
print("KNN Accuracy:", accuracy)

KNN Accuracy: 0.9046920821114369
